In [1]:
import pandas as pd

In [2]:
files = [
    "Cherkasy.csv", "Chernihiv.csv", "Chernivtsi.csv", "Dnipro.csv", "Ivano-Frankivsk.csv",
    "Kharkiv.csv", "Kherson.csv", "Khmelnytskyi.csv", "Kropyvnytskyi.csv",
    "Lutsk.csv", "Lviv.csv", "Mykolaiv.csv", "Odesa.csv", "Poltava.csv", "Rivne.csv",
    "Sumy.csv", "Ternopil.csv", "Uzhhorod.csv", "Vinnytsia.csv", "Zaporizhzhia.csv",
    "Zhytomyr.csv"
]

dfs = []
for filename in files:
    df = pd.read_csv(filename)
    dfs.append(df)
    print(f"✓ {filename}")

df_ukraine = pd.concat(dfs, ignore_index=True)

✓ Cherkasy.csv
✓ Chernihiv.csv
✓ Chernivtsi.csv
✓ Dnipro.csv
✓ Ivano-Frankivsk.csv
✓ Kharkiv.csv
✓ Kherson.csv
✓ Khmelnytskyi.csv
✓ Kropyvnytskyi.csv
✓ Lutsk.csv
✓ Lviv.csv
✓ Mykolaiv.csv
✓ Odesa.csv
✓ Poltava.csv
✓ Rivne.csv
✓ Sumy.csv
✓ Ternopil.csv
✓ Uzhhorod.csv
✓ Vinnytsia.csv
✓ Zaporizhzhia.csv
✓ Zhytomyr.csv


In [3]:
df_ukraine.info()

<class 'pandas.DataFrame'>
RangeIndex: 9865 entries, 0 to 9864
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   num_of_rooms       9865 non-null   int64  
 1   freshly_renovated  9800 non-null   object 
 2   area               9804 non-null   float64
 3   living_area        6962 non-null   float64
 4   kitchen_area       9789 non-null   float64
 5   floor              9856 non-null   float64
 6   floors_in_house    9856 non-null   float64
 7   year_of_building   6827 non-null   float64
 8   price              9848 non-null   float64
 9   house_type         9865 non-null   str    
 10  heating            9865 non-null   str    
 11  wall_type          9865 non-null   str    
 12  url                9865 non-null   str    
 13  district           9865 non-null   str    
 14  city               9865 non-null   str    
 15  geo_region         9865 non-null   str    
dtypes: float64(7), int64(1), object(1),

In [4]:
df_kyiv = pd.read_csv("Kyiv.csv")

In [5]:
df_kyiv.info()

<class 'pandas.DataFrame'>
RangeIndex: 7775 entries, 0 to 7774
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   num_of_rooms       7774 non-null   float64
 1   freshly_renovated  7678 non-null   object 
 2   area               7754 non-null   float64
 3   living_area        4412 non-null   float64
 4   kitchen_area       7744 non-null   float64
 5   floor              7768 non-null   float64
 6   floors_in_house    7768 non-null   float64
 7   year_of_building   7676 non-null   float64
 8   price              7771 non-null   float64
 9   house_type         7775 non-null   str    
 10  heating            7775 non-null   str    
 11  wall_type          7775 non-null   str    
 12  url                7775 non-null   str    
 13  district           7775 non-null   str    
 14  city               7775 non-null   str    
 15  geo_region         7775 non-null   str    
dtypes: float64(8), object(1), str(7)
me

In [6]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

NUMERIC_TARGETS = [
    "num_of_rooms", "area", "living_area", "kitchen_area",
    "floor", "floors_in_house", "year_of_building", "price",
]

ROUND_TO_INT = ["num_of_rooms", "floor", "floors_in_house", "year_of_building"]

CATEGORICAL_COMPLETE = ["house_type", "heating", "wall_type"]

PASSTHROUGH_UNTOUCHED = [
    "url",
    "is_low_floor", "is_high_floor", "is_middle_floor",
    "is_large_city", "is_close_to_eu", "is_near_border_threat", "is_tourist_hub",
]


def knn_impute_apartments(
    df: pd.DataFrame,
    location_cols: list[str],
    n_neighbors: int = 5,
) -> pd.DataFrame:
    df = df.copy()

    passthrough_cols = [c for c in PASSTHROUGH_UNTOUCHED if c in df.columns]
    passthrough = df[passthrough_cols].copy()

    fr = df["freshly_renovated"].map({True: 1.0, False: 0.0})

    cat_cols = [c for c in CATEGORICAL_COMPLETE if c in df.columns] + location_cols
    onehot = pd.get_dummies(df[cat_cols], columns=cat_cols, dummy_na=False)

    numeric_targets = [c for c in NUMERIC_TARGETS if c in df.columns]

    feature_df = pd.concat(
        [df[numeric_targets], fr.rename("freshly_renovated"), onehot],
        axis=1,
    )

    scaler = StandardScaler()
    scaled = scaler.fit_transform(feature_df)

    imputer = KNNImputer(n_neighbors=n_neighbors, weights="distance")
    imputed_scaled = imputer.fit_transform(scaled)

    imputed = pd.DataFrame(
        scaler.inverse_transform(imputed_scaled),
        columns=feature_df.columns,
        index=feature_df.index,
    )

    result = df.copy()
    for col in numeric_targets:
        result[col] = imputed[col]
    for col in ROUND_TO_INT:
        if col in result.columns:
            result[col] = result[col].round().astype("Int64")

    result["freshly_renovated"] = imputed["freshly_renovated"].round().astype(int).astype(bool)

    for col in passthrough_cols:
        result[col] = passthrough[col]

    return result


if __name__ == "__main__":

    df_kyiv = knn_impute_apartments(df_kyiv, location_cols=["district"])
    df_ukraine = knn_impute_apartments(df_ukraine, location_cols=["district", "city"])

In [7]:
PRICE_COLS = ["price"]
AREA_COLS = ["area", "living_area", "kitchen_area"]


def format_price_and_areas(
    df: pd.DataFrame,
    as_string: bool = False,
    thousands_sep: str = " ",
) -> pd.DataFrame:
    
    df = df.copy()

    for col in PRICE_COLS:
        if col not in df.columns:
            continue
        df[col] = df[col].round(0)
        if as_string:
            df[col] = df[col].apply(
                lambda x: f"{x:,.0f}".replace(",", thousands_sep) if pd.notna(x) else x
            )

    for col in AREA_COLS:
        if col not in df.columns:
            continue
        df[col] = df[col].round(1)
        if as_string:
            df[col] = df[col].apply(lambda x: f"{x:.1f}" if pd.notna(x) else x)

    return df


if __name__ == "__main__":

    df_kyiv = format_price_and_areas(df_kyiv)
    df_ukraine = format_price_and_areas(df_ukraine)

In [8]:
df_ukraine.info()

<class 'pandas.DataFrame'>
RangeIndex: 9865 entries, 0 to 9864
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   num_of_rooms       9865 non-null   Int64  
 1   freshly_renovated  9865 non-null   bool   
 2   area               9865 non-null   float64
 3   living_area        9865 non-null   float64
 4   kitchen_area       9865 non-null   float64
 5   floor              9865 non-null   Int64  
 6   floors_in_house    9865 non-null   Int64  
 7   year_of_building   9865 non-null   Int64  
 8   price              9865 non-null   float64
 9   house_type         9865 non-null   str    
 10  heating            9865 non-null   str    
 11  wall_type          9865 non-null   str    
 12  url                9865 non-null   str    
 13  district           9865 non-null   str    
 14  city               9865 non-null   str    
 15  geo_region         9865 non-null   str    
dtypes: Int64(4), bool(1), float64(4), s

In [9]:
df_ukraine['living_ratio'] = df_ukraine['living_area'] / df_ukraine['area']

large_cities = ['Lviv', 'Odesa', 'Dnipro', 'Kharkiv']
df_ukraine['is_large_city'] = df_ukraine['city'].isin(large_cities).astype(int)

border_eu_cities = ['Lviv', 'Uzhhorod', 'Ivano-Frankivsk', 'Lutsk', 'Chernivtsi', 'Ternopil', 'Rivne']
df_ukraine['is_close_to_eu'] = df_ukraine['city'].isin(border_eu_cities).astype(int)

border_risk_cities = ['Sumy', 'Chernihiv', 'Kharkiv', 'Zhytomyr', 'Lutsk', 'Rivne']
df_ukraine['is_near_border_threat'] = df_ukraine['city'].isin(border_risk_cities).astype(int)

tourist_cities = ['Lviv', 'Odesa', 'Ivano-Frankivsk', 'Uzhhorod', 'Chernivtsi']
df_ukraine['is_tourist_hub'] = df_ukraine['city'].isin(tourist_cities).astype(int)

df_ukraine['is_low_floor'] = 0
df_ukraine['is_high_floor'] = 0

high_mask = df_ukraine['floors_in_house'] > 6
df_ukraine.loc[high_mask & (df_ukraine['floor'] <= 3), 'is_low_floor'] = 1
df_ukraine.loc[high_mask & (df_ukraine['floor'] > (df_ukraine['floors_in_house'] - 3)), 'is_high_floor'] = 1

low_mask = df_ukraine['floors_in_house'] <= 6
df_ukraine.loc[low_mask & (df_ukraine['floor'] == 1), 'is_low_floor'] = 1
df_ukraine.loc[low_mask & (df_ukraine['floor'] == df_ukraine['floors_in_house']), 'is_high_floor'] = 1

In [10]:
df_kyiv['living_ratio'] = df_kyiv['living_area'] / df_kyiv['area']

df_kyiv['is_low_floor'] = 0
df_kyiv['is_high_floor'] = 0
df_kyiv['is_middle_floor'] = 0

high_building_mask = df_kyiv['floors_in_house'] > 6

df_kyiv.loc[high_building_mask & (df_kyiv['floor'] <= 3), 'is_low_floor'] = 1
df_kyiv.loc[high_building_mask & (df_kyiv['floor'] > (df_kyiv['floors_in_house'] - 3)), 'is_high_floor'] = 1
df_kyiv.loc[high_building_mask & (df_kyiv['floor'] > 3) & (df_kyiv['floor'] <= (df_kyiv['floors_in_house'] - 3)), 'is_middle_floor'] = 1

low_building_mask = df_kyiv['floors_in_house'] <= 6

df_kyiv.loc[low_building_mask & (df_kyiv['floor'] == 1), 'is_low_floor'] = 1
df_kyiv.loc[low_building_mask & (df_kyiv['floor'] == df_kyiv['floors_in_house']), 'is_high_floor'] = 1
df_kyiv.loc[low_building_mask & (df_kyiv['floor'] > 1) & (df_kyiv['floor'] < df_kyiv['floors_in_house']), 'is_middle_floor'] = 1

In [11]:
df_ukraine.drop_duplicates(ignore_index=True, inplace=True)
df_kyiv.drop_duplicates(ignore_index=True, inplace=True)

In [12]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler


def find_best_k(df: pd.DataFrame, col: str = "year_of_building", k_range=range(2, 9)) -> None:
    X = df[[col]].values.astype(float)
    Xs = StandardScaler().fit_transform(X)
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(Xs)
        sil = silhouette_score(Xs, km.labels_)
        print(f"k={k}: silhouette={sil:.3f}, inertia={km.inertia_:.1f}")


def cluster_by_year(df: pd.DataFrame, n_clusters: int = 3, col: str = "year_of_building") -> pd.DataFrame:
    df = df.copy()
    X = df[[col]].values.astype(float)
    Xs = StandardScaler().fit_transform(X)
    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10).fit(Xs)
    df["year_cluster"] = km.labels_
    return df


def summarize_clusters(df: pd.DataFrame, col: str = "year_of_building") -> pd.DataFrame:
    return (
        df.groupby("year_cluster")
        .agg(
            n=(col, "size"),
            year_min=(col, "min"),
            year_max=(col, "max"),
            year_mean=(col, "mean"),
            price_mean=("price", "mean"),
            area_mean=("area", "mean"),
        )
        .sort_values("year_min")
    )


if __name__ == "__main__":
    for df, name in [(df_ukraine, "Ukraine"), (df_kyiv, "Kyiv")]:
        df = df

        print(f"=== {name}: підбір k ===")
        find_best_k(df)

        df_clustered = cluster_by_year(df, n_clusters=3)
        print(f"\n=== {name}: підсумок по кластерах (k=3) ===")
        print(summarize_clusters(df_clustered))
        print()

=== Ukraine: підбір k ===
k=2: silhouette=0.784, inertia=2457.4
k=3: silhouette=0.802, inertia=1182.3
k=4: silhouette=0.802, inertia=829.3
k=5: silhouette=0.660, inertia=588.7
k=6: silhouette=0.568, inertia=420.9
k=7: silhouette=0.602, inertia=306.4
k=8: silhouette=0.595, inertia=231.3

=== Ukraine: підсумок по кластерах (k=3) ===
                 n  year_min  year_max    year_mean   price_mean  area_mean
year_cluster                                                               
2              311      1507      1937  1905.337621  1354.800643  76.553376
0             2900      1938      1993  1969.345862   810.776552  49.898897
1             6654      1994      2028  2018.274121  1433.212654  72.583980

=== Kyiv: підбір k ===
k=2: silhouette=0.812, inertia=1619.7
k=3: silhouette=0.819, inertia=627.7
k=4: silhouette=0.665, inertia=348.6
k=5: silhouette=0.646, inertia=242.2
k=6: silhouette=0.648, inertia=190.1
k=7: silhouette=0.608, inertia=138.4
k=8: silhouette=0.608, inertia=110.2

==

In [13]:
df_ukraine["is_historic"] = (df_ukraine["year_of_building"] <= 1937).astype(int)
df_ukraine["is_soviet"] = ((df_ukraine["year_of_building"] >= 1938) & (df_ukraine["year_of_building"] <= 1993)).astype(int)

In [14]:
df_kyiv["is_historic"] = (df_kyiv["year_of_building"] <= 1938).astype(int)
df_kyiv["is_soviet"] = ((df_kyiv["year_of_building"] >= 1939) & (df_kyiv["year_of_building"] <= 1992)).astype(int)

In [15]:
df_ukraine.to_csv("Ukraine_for_analysis.csv", index=False)
df_kyiv.to_csv("Kyiv_for_analysis.csv", index=False)

In [16]:
df_ukraine.info()

<class 'pandas.DataFrame'>
RangeIndex: 9865 entries, 0 to 9864
Data columns (total 25 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   num_of_rooms           9865 non-null   Int64  
 1   freshly_renovated      9865 non-null   bool   
 2   area                   9865 non-null   float64
 3   living_area            9865 non-null   float64
 4   kitchen_area           9865 non-null   float64
 5   floor                  9865 non-null   Int64  
 6   floors_in_house        9865 non-null   Int64  
 7   year_of_building       9865 non-null   Int64  
 8   price                  9865 non-null   float64
 9   house_type             9865 non-null   str    
 10  heating                9865 non-null   str    
 11  wall_type              9865 non-null   str    
 12  url                    9865 non-null   str    
 13  district               9865 non-null   str    
 14  city                   9865 non-null   str    
 15  geo_region     

In [17]:
df_ukraine.drop(["url"], axis=1, inplace=True)
df_kyiv.drop(["url"], axis=1, inplace=True)

In [18]:
df_ukraine = df_ukraine[df_ukraine["price"].notnull()]
df_kyiv = df_kyiv[df_kyiv["price"].notnull()]

In [19]:
df_ukraine.info()

<class 'pandas.DataFrame'>
RangeIndex: 9865 entries, 0 to 9864
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   num_of_rooms           9865 non-null   Int64  
 1   freshly_renovated      9865 non-null   bool   
 2   area                   9865 non-null   float64
 3   living_area            9865 non-null   float64
 4   kitchen_area           9865 non-null   float64
 5   floor                  9865 non-null   Int64  
 6   floors_in_house        9865 non-null   Int64  
 7   year_of_building       9865 non-null   Int64  
 8   price                  9865 non-null   float64
 9   house_type             9865 non-null   str    
 10  heating                9865 non-null   str    
 11  wall_type              9865 non-null   str    
 12  district               9865 non-null   str    
 13  city                   9865 non-null   str    
 14  geo_region             9865 non-null   str    
 15  living_ratio   

In [20]:
df_kyiv.info()

<class 'pandas.DataFrame'>
RangeIndex: 7775 entries, 0 to 7774
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   num_of_rooms       7775 non-null   Int64  
 1   freshly_renovated  7775 non-null   bool   
 2   area               7775 non-null   float64
 3   living_area        7775 non-null   float64
 4   kitchen_area       7775 non-null   float64
 5   floor              7775 non-null   Int64  
 6   floors_in_house    7775 non-null   Int64  
 7   year_of_building   7775 non-null   Int64  
 8   price              7775 non-null   float64
 9   house_type         7775 non-null   str    
 10  heating            7775 non-null   str    
 11  wall_type          7775 non-null   str    
 12  district           7775 non-null   str    
 13  city               7775 non-null   str    
 14  geo_region         7775 non-null   str    
 15  living_ratio       7775 non-null   float64
 16  is_low_floor       7775 non-null   

In [21]:
df_ukraine = df_ukraine[df_ukraine["freshly_renovated"].notnull()]
df_kyiv = df_kyiv[df_kyiv["freshly_renovated"].notnull()]

In [22]:
cols_to_clean = ["price", "area"]

def remove_outliers_iqr(group):
    clean_group = group.copy()
    
    for col in cols_to_clean:
        Q1 = clean_group[col].quantile(0.25)
        Q3 = clean_group[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        clean_group = clean_group[(clean_group[col] >= lower_bound) & (clean_group[col] <= upper_bound)]
        
    return clean_group

df_ukraine_clean = remove_outliers_iqr(df_ukraine)
df_kyiv_clean = remove_outliers_iqr(df_kyiv)

In [23]:
df_ukraine_clean.describe()

,num_of_rooms,area,living_area,kitchen_area,floor,floors_in_house,year_of_building,price,living_ratio,is_large_city,is_close_to_eu,is_near_border_threat,is_tourist_hub,is_low_floor,is_high_floor,is_historic,is_soviet
count,9021.0,9021.000000,9021.000000,9021.000000,9021.0,9021.0,9021.0,9021.000000,9021.000000,9021.000000,9021.000000,9021.000000,9021.000000,9021.000000,9021.000000,9021.000000,9021.000000
mean,1.88039,58.484569,31.377009,13.409899,6.228024,10.160071,1999.88039,1173.099878,0.545674,0.683073,0.298193,0.191886,0.534531,0.189558,0.318701,0.028600,0.317925
std,0.819655,22.249213,13.826537,7.684320,4.787015,6.162605,29.289593,528.682321,0.154786,0.465305,0.457490,0.393805,0.498834,0.391973,0.465999,0.166689,0.465696
min,1.0,8.600000,1.000000,1.000000,1.0,1.0,1507.0,24.000000,0.008621,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.0,43.000000,20.000000,7.000000,3.0,5.0,1973.0,782.000000,0.437500,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2.0,54.200000,30.000000,12.000000,5.0,9.0,2015.0,1044.000000,0.542857,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
75%,2.0,72.000000,39.900000,17.000000,9.0,13.0,2022.0,1500.000000,0.642570,1.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000
max,6.0,125.100000,110.300000,99.000000,28.0,36.0,2028.0,2753.000000,2.025000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [24]:
df_ukraine_clean = df_ukraine_clean[df_ukraine_clean['kitchen_area'] < df_ukraine_clean['area']]
df_ukraine_clean = df_ukraine_clean[df_ukraine_clean['living_area'] < df_ukraine_clean['area']]
df_ukraine_clean = df_ukraine_clean[(df_ukraine_clean['living_area'] + df_ukraine_clean['kitchen_area']) <= df_ukraine_clean['area']]

df_ukraine_clean = df_ukraine_clean[df_ukraine_clean['floors_in_house'] <= 50]
df_ukraine_clean = df_ukraine_clean[df_ukraine_clean['floor'] <= df_ukraine_clean['floors_in_house']]

df_ukraine_clean = df_ukraine_clean[df_ukraine_clean['area'] >= 15]
df_ukraine_clean = df_ukraine_clean[df_ukraine_clean['living_area'] >= 9]
df_ukraine_clean = df_ukraine_clean[df_ukraine_clean['kitchen_area'] >= 4]
df_ukraine_clean = df_ukraine_clean[df_ukraine_clean['price'] >= 200]

df_ukraine_clean.describe()

,num_of_rooms,area,living_area,kitchen_area,floor,floors_in_house,year_of_building,price,living_ratio,is_large_city,is_close_to_eu,is_near_border_threat,is_tourist_hub,is_low_floor,is_high_floor,is_historic,is_soviet
count,8528.0,8528.000000,8528.000000,8528.000000,8528.0,8528.0,8528.0,8528.000000,8528.000000,8528.000000,8528.000000,8528.000000,8528.000000,8528.000000,8528.000000,8528.000000,8528.000000
mean,1.910413,59.672338,31.536574,13.533408,6.297608,10.273335,2000.157833,1183.117026,0.531083,0.682927,0.308513,0.181051,0.553119,0.187148,0.319418,0.028494,0.312383
std,0.818712,21.780182,13.677702,7.404685,4.807039,6.177943,29.310507,527.856221,0.130314,0.465363,0.461907,0.385083,0.497200,0.390053,0.466279,0.166390,0.463492
min,1.0,15.000000,9.000000,4.000000,1.0,1.0,1507.0,200.000000,0.128205,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.0,43.800000,20.000000,7.300000,3.0,5.0,1974.0,793.750000,0.432416,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2.0,55.700000,30.000000,12.000000,5.0,9.0,2015.0,1052.000000,0.533333,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
75%,2.0,72.500000,40.000000,17.000000,9.0,14.0,2022.0,1511.250000,0.627907,1.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000
max,6.0,125.100000,100.000000,68.000000,28.0,36.0,2028.0,2753.000000,0.909091,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [25]:
df_ukraine_clean.info()

<class 'pandas.DataFrame'>
Index: 8528 entries, 0 to 9863
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   num_of_rooms           8528 non-null   Int64  
 1   freshly_renovated      8528 non-null   bool   
 2   area                   8528 non-null   float64
 3   living_area            8528 non-null   float64
 4   kitchen_area           8528 non-null   float64
 5   floor                  8528 non-null   Int64  
 6   floors_in_house        8528 non-null   Int64  
 7   year_of_building       8528 non-null   Int64  
 8   price                  8528 non-null   float64
 9   house_type             8528 non-null   str    
 10  heating                8528 non-null   str    
 11  wall_type              8528 non-null   str    
 12  district               8528 non-null   str    
 13  city                   8528 non-null   str    
 14  geo_region             8528 non-null   str    
 15  living_ratio        

In [26]:
df_kyiv_clean.info()

<class 'pandas.DataFrame'>
Index: 7092 entries, 0 to 7773
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   num_of_rooms       7092 non-null   Int64  
 1   freshly_renovated  7092 non-null   bool   
 2   area               7092 non-null   float64
 3   living_area        7092 non-null   float64
 4   kitchen_area       7092 non-null   float64
 5   floor              7092 non-null   Int64  
 6   floors_in_house    7092 non-null   Int64  
 7   year_of_building   7092 non-null   Int64  
 8   price              7092 non-null   float64
 9   house_type         7092 non-null   str    
 10  heating            7092 non-null   str    
 11  wall_type          7092 non-null   str    
 12  district           7092 non-null   str    
 13  city               7092 non-null   str    
 14  geo_region         7092 non-null   str    
 15  living_ratio       7092 non-null   float64
 16  is_low_floor       7092 non-null   int64

In [27]:
df_kyiv_clean = df_kyiv_clean[df_kyiv_clean['kitchen_area'] < df_kyiv_clean['area']]
df_kyiv_clean = df_kyiv_clean[df_kyiv_clean['living_area'] < df_kyiv_clean['area']]
df_kyiv_clean = df_kyiv_clean[(df_kyiv_clean['living_area'] + df_kyiv_clean['kitchen_area']) <= df_kyiv_clean['area']]

df_kyiv_clean = df_kyiv_clean[df_kyiv_clean['floors_in_house'] <= 50]
df_kyiv_clean = df_kyiv_clean[df_kyiv_clean['floor'] <= df_kyiv_clean['floors_in_house']]

df_kyiv_clean = df_kyiv_clean[df_kyiv_clean['area'] >= 15]
df_kyiv_clean = df_kyiv_clean[df_kyiv_clean['living_area'] >= 9]
df_kyiv_clean = df_kyiv_clean[df_kyiv_clean['kitchen_area'] >= 4]
df_kyiv_clean = df_kyiv_clean[df_kyiv_clean['price'] >= 200]

df_kyiv_clean.describe()

,num_of_rooms,area,living_area,kitchen_area,floor,floors_in_house,year_of_building,price,living_ratio,is_low_floor,is_high_floor,is_middle_floor,is_historic,is_soviet
count,6773.0,6773.000000,6773.000000,6773.000000,6773.0,6773.0,6773.0,6773.000000,6773.000000,6773.000000,6773.000000,6773.000000,6773.000000,6773.000000
mean,1.969437,67.077396,33.106172,15.161007,10.675329,18.197401,2005.523992,1798.458733,0.492254,0.123284,0.222353,0.654511,0.029234,0.179389
std,0.896824,28.486696,16.788493,7.810417,7.380742,8.734425,26.474679,617.852695,0.118545,0.328787,0.415858,0.475563,0.168473,0.383706
min,1.0,17.400000,9.000000,4.000000,1.0,1.0,1858.0,481.000000,0.146341,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.0,44.300000,19.000000,9.800000,4.0,9.0,2005.0,1339.000000,0.403846,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2.0,61.300000,30.000000,14.000000,9.0,22.0,2017.0,1667.000000,0.486486,0.000000,0.000000,1.000000,0.000000,0.000000
75%,3.0,84.400000,42.000000,19.000000,16.0,25.0,2022.0,2194.000000,0.571924,0.000000,0.000000,1.000000,0.000000,0.000000
max,6.0,154.000000,125.000000,73.900000,36.0,38.0,2028.0,3710.000000,0.882353,1.000000,1.000000,1.000000,1.000000,1.000000


In [28]:
df_kyiv_clean.info()

<class 'pandas.DataFrame'>
Index: 6773 entries, 0 to 7773
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   num_of_rooms       6773 non-null   Int64  
 1   freshly_renovated  6773 non-null   bool   
 2   area               6773 non-null   float64
 3   living_area        6773 non-null   float64
 4   kitchen_area       6773 non-null   float64
 5   floor              6773 non-null   Int64  
 6   floors_in_house    6773 non-null   Int64  
 7   year_of_building   6773 non-null   Int64  
 8   price              6773 non-null   float64
 9   house_type         6773 non-null   str    
 10  heating            6773 non-null   str    
 11  wall_type          6773 non-null   str    
 12  district           6773 non-null   str    
 13  city               6773 non-null   str    
 14  geo_region         6773 non-null   str    
 15  living_ratio       6773 non-null   float64
 16  is_low_floor       6773 non-null   int64

In [29]:
df_kyiv_clean.drop(["city", "geo_region"], axis=1, inplace=True)

In [30]:
categorical_cols_kyiv = ['district', 'freshly_renovated', 'house_type', 'heating', 'wall_type'] 

df_kyiv_encoded = pd.get_dummies(df_kyiv_clean, columns=categorical_cols_kyiv, drop_first=True, dtype=int)

print("Колонки київської моделі після OHE:")
print(df_kyiv_encoded.columns.tolist())

Колонки київської моделі після OHE:
['num_of_rooms', 'area', 'living_area', 'kitchen_area', 'floor', 'floors_in_house', 'year_of_building', 'price', 'living_ratio', 'is_low_floor', 'is_high_floor', 'is_middle_floor', 'is_historic', 'is_soviet', 'district_Desnianskyi', 'district_Dniprovskyi', 'district_Holosiivskyi', 'district_Obolonskyi', 'district_Pecherskyi', 'district_Podilskyi', 'district_Shevchenkivskyi', 'district_Solomianskyi', 'district_Sviatoshynskyi', 'freshly_renovated_True', 'house_type_khrushchivka', 'house_type_new_build', 'house_type_pre_revolutionary', 'house_type_stalinka', 'heating_centralized', 'heating_individual', 'wall_type_brick', 'wall_type_insulated_panel', 'wall_type_monolithic_frame', 'wall_type_panel']


In [31]:
categorical_cols_regions = ['city', 'district', 'freshly_renovated', 'geo_region', 'house_type', 'heating', 'wall_type']

df_ukraine_encoded = pd.get_dummies(df_ukraine_clean, columns=categorical_cols_regions, drop_first=True, dtype=int)


print("\nКолонки моделі регіонів після OHE:")
print(df_ukraine_encoded.columns.tolist())


Колонки моделі регіонів після OHE:
['num_of_rooms', 'area', 'living_area', 'kitchen_area', 'floor', 'floors_in_house', 'year_of_building', 'price', 'living_ratio', 'is_large_city', 'is_close_to_eu', 'is_near_border_threat', 'is_tourist_hub', 'is_low_floor', 'is_high_floor', 'is_historic', 'is_soviet', 'city_Chernihiv', 'city_Chernivtsi', 'city_Dnipro', 'city_Ivano-Frankivsk', 'city_Kharkiv', 'city_Kherson', 'city_Khmelnytskyi', 'city_Kropyvnytskyi', 'city_Lutsk', 'city_Lviv', 'city_Mykolaiv', 'city_Odesa', 'city_Poltava', 'city_Rivne', 'city_Sumy', 'city_Ternopil', 'city_Uzhhorod', 'city_Vinnytsia', 'city_Zaporizhzhia', 'city_Zhytomyr', 'district_Outskirts', 'district_Residential', 'freshly_renovated_True', 'geo_region_East', 'geo_region_North', 'geo_region_South', 'geo_region_West', 'house_type_khrushchivka', 'house_type_new_build', 'house_type_pre_revolutionary', 'house_type_stalinka', 'heating_centralized', 'heating_individual', 'wall_type_brick', 'wall_type_insulated_panel', 'wall

In [32]:
df_ukraine_encoded.to_csv("Ukraine.csv", index=False)
df_kyiv_encoded.to_csv("Kyiv_ML.csv", index=False)